# 🌱 Aula 19 — Otimização de Processamento e Uso de Energia em GPUs

**Curso:** Técnico em Inteligência Artificial | **UC:** Introdução a Arquitetura de Computadores

---

## 🎯 Situação de Aprendizagem

O data center da instituição recebeu uma notificação de custo energético: o cluster GPU consome **40% mais energia** do que o projetado. Uma auditoria revelou que as GPUs operam no **TDP máximo** mesmo durante fases de baixa carga do treinamento. O time precisa implementar um sistema automático de gerenciamento de potência: medir o consumo real, calibrar **Power Limits** por fase de treinamento e criar alertas térmicos — tudo via Bash e Python com `nvidia-smi` e `nvidia-ml-py`.

Nesta aula vamos **integrar** os três pilares do Bloco 3:

1. **Automação** — coleta contínua de telemetria (Aula 17);
2. **Gestão** — fila e exclusão mútua na GPU (Aula 18);
3. **Energia** — Power Limit, térmica e eficiência (esta aula).

In [ ]:
# 1. Verificar GPU e ferramentas disponíveis no ambiente (Colab ou local)
# Se não houver GPU NVIDIA, todo o notebook roda em MODO SIMULADO.
import shutil, subprocess

TEM_NVIDIA_SMI = shutil.which("nvidia-smi") is not None
print(f"nvidia-smi disponível: {TEM_NVIDIA_SMI}")

if TEM_NVIDIA_SMI:
    # Lista as GPUs e mostra o consumo/limite atuais
    print(subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,power.draw,power.limit,temperature.gpu",
         "--format=csv,noheader"], capture_output=True, text=True).stdout)
else:
    # Sem GPU: seguimos com dados sintéticos para exercitar os conceitos
    print("NVIDIA GPU não detectada. Executando em MODO SIMULADO.")

## 1. Monitoramento térmico e energético

O mesmo monitor contínuo da Aula 17 ganha, agora, colunas de **potência** e **temperatura** — a base para medir eficiência e disparar alertas.

In [ ]:
%%writefile monitor_thermal.py
# 2. Coletor térmico/energético: salva um CSV com potência, temperatura e clock
# monitor_thermal.py — coleta temperatura, potência e clock da GPU
# Uso: python3 monitor_thermal.py [intervalo_s] [amostras] [arquivo_csv]
import csv, os, random, shutil, subprocess, sys, time

INTERVALO = int(sys.argv[1]) if len(sys.argv) > 1 else 1
AMOSTRAS  = int(sys.argv[2]) if len(sys.argv) > 2 else 10
SAIDA     = sys.argv[3] if len(sys.argv) > 3 else "gpu_thermal.csv"
TEM_SMI   = shutil.which("nvidia-smi") is not None

CABECALHO = ["timestamp", "gpu_idx", "temp_c", "power_w", "power_limit_w",
             "util_gpu_pct", "clock_sm_mhz", "clock_mem_mhz"]

with open(SAIDA, "w", newline="", encoding="utf-8") as f:
    escritor = csv.writer(f)
    escritor.writerow(CABECALHO)

    for i in range(AMOSTRAS):
        ts = time.strftime("%Y-%m-%d %H:%M:%S")
        if TEM_SMI:
            # Consulta real: um ponto por GPU
            saida = subprocess.check_output([
                "nvidia-smi",
                "--query-gpu=index,temperature.gpu,power.draw,enforced.power.limit,"
                "utilization.gpu,clocks.sm,clocks.mem",
                "--format=csv,noheader,nounits"], text=True).strip()
            for linha in saida.split("\n"):
                campos = [c.strip() for c in linha.split(",")]
                escritor.writerow([ts] + campos)
                print(f"  [{ts}] GPU{campos[0]} | {campos[1]}C | {campos[2]}W/{campos[3]}W")
        else:
            # Modo simulado: gera um valor plausível de uma GPU em treino
            temp = 55 + random.randint(0, 30)
            pwr  = 120 + random.randint(0, 130)
            util = 60 + random.randint(0, 40)
            escritor.writerow([ts, 0, temp, pwr, 250, util, 1410, 5001])
            print(f"  [{ts}] GPU0 | {temp}C | {pwr}W/250W | util {util}%")
        time.sleep(INTERVALO)

print(f"\nColeta concluída -> {SAIDA}")


In [ ]:
# Executa o coletor por 5 amostras (rápido) e mostra o resultado
!python3 monitor_thermal.py 1 5 gpu_thermal.csv
!cat gpu_thermal.csv

## 2. Power Limit: medindo o trade-off

Ajustar o Power Limit exige privilégio de root e GPU NVIDIA real. Na ausência disso, **simulamos o trade-off** com um modelo empírico: ao reduzir o PL, o throughput cai um pouco e o consumo cai bastante — melhorando a **eficiência (imgs/J)** até um ponto ótimo.

In [ ]:
# 3. Definir Power Limit (real se houver GPU; senão apenas registra a intenção)
def set_power_limit(watts, gpu=0):
    """Aplica o Power Limit via nvidia-smi. Requer sudo/root em GPU real."""
    if not TEM_NVIDIA_SMI:
        print(f"[SIMULADO] Power Limit da GPU{gpu} definido para {watts}W")
        return
    try:
        subprocess.run(["sudo", "nvidia-smi", "-i", str(gpu), "-pl", str(watts)], check=True)
        print(f"GPU{gpu}: Power Limit = {watts}W")
    except Exception as e:
        print(f"Não foi possível aplicar (precisa de root?): {e}")

# Exemplo de uso
set_power_limit(180)

In [ ]:
# 4. Benchmark energético SIMULADO: throughput e eficiência por Power Limit
# Modelo logístico: PL muito baixo sofre throttling severo; acima de ~60% do
# TDP o ganho satura. Assim a eficiência (imgs/J) tem ponto ótimo ~70-75% do TDP.
import math

TDP = 250  # potência máxima de referência da GPU (W)

def throughput_imgs_s(pl, tdp=TDP):
    # Curve logística em função da fração do TDP
    x = pl / tdp
    return 480 / (1 + math.exp(-6.0 * (x - 0.5)))

resultados = []
print(f"{'PL (W)':>8} {'Imgs/s':>10} {'Pwr (W)':>10} {'Imgs/J':>10}")
print("-" * 42)
for pl in [100, 150, 200, 225, 250]:
    imgs_s = throughput_imgs_s(pl)
    pwr    = pl * 0.92                      # consumo médio ≈ 92% do limite
    efic   = imgs_s / pwr
    resultados.append((pl, imgs_s, pwr, efic))
    print(f"{pl:>8} {imgs_s:>10.1f} {pwr:>10.1f} {efic:>10.3f}")

melhor = max(resultados, key=lambda r: r[3])
print(f"\nPonto ótimo de eficiência: PL={melhor[0]}W (~{melhor[0]/TDP:.0%} do TDP), {melhor[3]:.3f} imgs/J")

In [ ]:
# 5. Gráfico: trade-off entre throughput e eficiência
import matplotlib.pyplot as plt

pls    = [r[0] for r in resultados]
imgs   = [r[1] for r in resultados]
efics  = [r[3] for r in resultados]

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(pls, imgs, "o-", color="#6366F1", label="Throughput (imgs/s)")
ax1.set_xlabel("Power Limit (W)")
ax1.set_ylabel("Throughput (imgs/s)", color="#6366F1")
ax1.tick_params(axis="y", labelcolor="#6366F1")

ax2 = ax1.twinx()
ax2.plot(pls, efics, "s--", color="#10B981", label="Eficiência (imgs/J)")
ax2.set_ylabel("Eficiência (imgs/J)", color="#10B981")
ax2.tick_params(axis="y", labelcolor="#10B981")

ax1.axvline(x=melhor[0], color="#F97316", linestyle=":", alpha=0.7)
ax1.set_title("Trade-off: Desempenho x Eficiência Energética")
plt.tight_layout()
plt.savefig("tradeoff_energia.png", dpi=120)
plt.show()

## 3. Controle programático com nvidia-ml-py (NVML)

O `nvidia-ml-py` permite ler e configurar a GPU **sem parsing de texto**. Em GPU real, ele acessa temperatura, potência, VRAM e clocks diretamente.

In [ ]:
# 6. Leitura via NVML (nvidia-ml-py). Sem GPU, mostra a estrutura esperada
try:
    import pynvml

    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)

    nome = pynvml.nvmlDeviceGetName(handle)
    temp = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)
    pwr  = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000          # mW -> W
    pl   = pynvml.nvmlDeviceGetPowerManagementLimit(handle) / 1000
    util = pynvml.nvmlDeviceGetUtilizationRates(handle)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)

    print(f"GPU:         {nome}")
    print(f"Temperatura: {temp}C")
    print(f"Potência:    {pwr:.1f}W / {pl:.1f}W")
    print(f"VRAM:        {info.used // 1024**2}MB / {info.total // 1024**2}MB")
    print(f"Utilização:  GPU={util.gpu}% | MEM={util.memory}%")

    pl_min, pl_max = pynvml.nvmlDeviceGetPowerManagementLimitConstraints(handle)
    print(f"Faixa de Power Limit suportada: {pl_min//1000}W - {pl_max//1000}W")
    pynvml.nvmlShutdown()
except Exception as e:
    # Sem GPU/NVML: exibimos os mesmos campos com valores de referência (T4)
    print("[SIMULADO] NVML indisponível:", e)
    print("GPU:         Tesla T4 (referência)")
    print("Temperatura: 67C")
    print("Potência:    68.5W / 70.0W")
    print("VRAM:        512MB / 15360MB")
    print("Utilização:  GPU=85% | MEM=42%")
    print("Faixa de Power Limit suportada: 60W - 70W")

## 4. Integração com gestão de processos (Aulas 17 e 18)

O gerenciamento de energia só faz sentido com **exclusão mútua**: um job por vez na GPU evita OOM e medições contaminadas. Abaixo, um lock portátil (por diretório) que funciona no Colab e no Windows.

In [ ]:
# 7. Exclusão mútua com lock por diretório (mkdir é atômico)
import os, time

LOCK_DIR = "gpu_exclusiva.lockdir"

def adquirir_lock(timeout=10):
    inicio = time.time()
    while True:
        try:
            os.mkdir(LOCK_DIR)      # atômico: falha se já existir
            print("Lock adquirido — GPU exclusiva para este job.")
            return True
        except FileExistsError:
            if time.time() - inicio > timeout:
                print("Timeout: GPU ocupada por outro processo.")
                return False
            time.sleep(0.5)

def liberar_lock():
    try:
        os.rmdir(LOCK_DIR)
        print("Lock liberado.")
    except FileNotFoundError:
        pass

# Demonstração: adquire, executa algo e libera
if adquirir_lock():
    print("  ... treinando (simulado) com Power Limit controlado ...")
    time.sleep(1)
    liberar_lock()

---
## 📝 5. Exercícios Práticos (5)

Resolva os 5 exercícios a seguir completando as células de código correspondentes.

### Exercício 1: Coleta com intervalo de 2 s

Execute o `monitor_thermal.py` coletando **8 amostras** a cada **2 segundos** em `gpu_ex1.csv` e exiba o arquivo.

In [ ]:
# Escreva a sua solução para o Exercício 1 abaixo:
# Exemplo: !python3 monitor_thermal.py 2 8 gpu_ex1.csv

### Exercício 2: Alerta térmico automático

Escreva um trecho Python que leia `gpu_thermal.csv` e imprima **CRÍTICO** para temperaturas ≥ 88 °C, **AVISO** para ≥ 80 °C e **OK** abaixo disso.

In [ ]:
# Escreva a sua solução para o Exercício 2 abaixo:
# import csv ... for row in ...: if int(row['temp_c']) >= 88: ...

### Exercício 3: Encontrar o PL de eficiência máxima

Usando o modelo do benchmark (`throughput_imgs_s`), varra os PLs de 100 a 250 W em passos de 10 e imprima qual deles tem a **maior eficiência (imgs/J)**.

In [ ]:
# Escreva a sua solução para o Exercício 3 abaixo:
# for pl in range(100, 251, 10): efic = throughput_imgs_s(pl)/(pl*0.92) ...

### Exercício 4: Ajuste dinâmico de PL por época

Simule 5 épocas de treino: use PL alto (250 W) nas 2 primeiras e PL reduzido (150 W) nas 3 finais. Imprima, a cada época, o PL aplicado e uma estimativa de throughput.

In [ ]:
# Escreva a sua solução para o Exercício 4 abaixo:
# for epoca in range(1, 6): pl = 250 if epoca <= 2 else 150 ...

### Exercício 5: Aplicar o lock de exclusão mútua

Reutilize `adquirir_lock`/`liberar_lock`: execute um "job" simulado de 2 s, garanta que o lock seja **sempre liberado** (mesmo em erro) e tente adquirir o lock uma segunda vez para provar que está livre.

In [ ]:
# Escreva a sua solução para o Exercício 5 abaixo:
# Dica: use try/finally para garantir liberar_lock().

---
## 📌 6. Síntese e Conclusão

- **`nvidia-smi -pl`** define o Power Limit em Watts — o controle energético mais eficaz.
- **Thermal Throttling** reduz o clock automaticamente acima do limite térmico; o PL evita chegar lá.
- **nvidia-ml-py (NVML)** dá acesso programático a temperatura, potência, VRAM e clocks.
- **Eficiência (imgs/J)** balanceia throughput e consumo — o ponto ótimo costuma ficar em ~75% do TDP.
- **Alertas via cron** monitoram periodicamente e podem agir sobre o PL automaticamente.
- **Exclusão mútua** (lock) garante medições confiáveis e evita OOM durante o treino.